In [ ]:
#importing required libraries
!pip3 install  langchain langchain_community chromadb  langchain-google-vertexai
!pip3 install "unstructured[all-docs]" pypdf pillow pydantic lxml pillow matplotlib chromadb tiktoken
#checking dependencies

In [ ]:
#initializing..
import os
import sys

from dotenv import load_dotenv
load_dotenv()

from config.auth import *
PROJECT_ID=os.getenv("PROJECT_ID")
REGION=os.getenv("REGION")

import vertexai 
vertexai.init(project=PROJECT_ID,location=REGION)

In [ ]:
#basic example
import logging 
import zipfile
import requests

# importing required configs
logging.basicConfig(level=logging.INFO)
DATA_URL=os.getenv("DATA_URL")

#requesting..
response=requests.get(DATA_URL)
filename='resources/resources.zip'
with open(filename,"wb") as file:
    file.write(response.content)



#extracting current zip filter
with zipfile.ZipFile(filename,"r") as zip_ref:
    zip_ref.extractall()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("resources/resources.zip")
docs=loader.load()
tables=[]
texts=[d.page_content for d in docs]

In [ ]:
#checking contents
print("First:/t",texts[0])
print("Length:/t",len(texts))

In [ ]:
##Generating text summaries
#importing required libraries
from langchain_google_vertexai import VertexAI , ChatVertexAI , VertexAIEmbeddings
# import vertexai 
from langchain.prompts import PromptTemplate
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

#Generate summaries of text elements
def summaryFromInput(texts,tables,promptText,summarizeTexts=False,modelName='gemini-3-flash-preview'):
    prompt=PromptTemplate.from_template(promptText)

    empty=RunnableLambda(
        lambda x:AIMessage(content="Error processing document")
    )

    ##evulating chain
    #current model
    model=VertexAI(
        temprature=0,model_name=modelName,max_output_token=1024

    ).with_fallbacks([empty])

    summary={"element":lambda x:x}|prompt|model|StrOutputParser()


    #summaries
    textSummaries=[]
    tableSummaries=[]

    if texts and summary:
        #updating
        textSummaries=summary.batch(texts,{"max_concurrency":1})
    elif texts:
        textSummaries=texts

    if tables:
        tableSummaries=summary.batch(tables,{"max_concurrency":1})
    return textSummaries,tableSummaries

In [ ]:
#testing current function
promptText=""" You are a scientific research assistant specializing in invasive species, plant protection, ecology, and plant pathology.

Your task is to analyze the provided document and generate a detailed, accurate, and comprehensive summary suitable for a Retrieval-Augmented Generation (RAG) system.

## Instructions

1. Read and understand the entire input document.
2. Identify the main subject, objectives, and important findings.
3. Preserve all scientifically relevant information.
4. Do not introduce external information or unsupported claims.
5. Maintain scientific terminology and Latin species names.
6. Preserve geographical locations, dates, numerical values, and experimental findings.
7. Distinguish confirmed scientific findings from hypotheses and assumptions.
8. Do not omit important details simply to make the summary shorter.
9. If information is missing, explicitly state that it is not specified.
10. Maintain references to the original document whenever available.

## Summary Structure

Organize the summary using the following sections:

### 1. General Overview
Explain the document's main topic, purpose, research objectives, and scope.

### 2. Invasive Species
Identify the invasive species discussed, including:
- Scientific and common names.
- Biological characteristics.
- Native and introduced distribution.
- Reproduction and spreading mechanisms.
- Ecological characteristics.

### 3. Host Plants
Identify affected plant species, including:
- Scientific and common names.
- Susceptible plant species.
- Plant parts affected.
- Host-specificity information.

### 4. Geographic Distribution
Summarize:
- Countries and regions where the species has been reported.
- Confirmed geographical presence.
- Potential distribution.
- Environmental conditions affecting distribution.

Clearly distinguish confirmed presence from predicted or potential distribution.

### 5. Symptoms and Damage
Describe:
- Visible infestation symptoms.
- Physical damage to plants.
- Ecological consequences.
- Economic consequences.
- Early detection indicators.

### 6. Prevention and Control Methods
Summarize scientifically supported recommendations:
- Preventive measures.
- Monitoring and early detection.
- Mechanical and cultural control.
- Biological control.
- Chemical control.
- Long-term management strategies.

Include application conditions and limitations whenever available.

### 7. Scientific Findings
Preserve important experimental results, statistics, observations, and research conclusions.

### 8. Limitations and Uncertainties
Identify research limitations, missing information, contradictory findings, and unresolved questions.

### 9. Keywords
Generate relevant keywords for semantic search, including scientific names, host plants, geographical locations, symptoms, and prevention methods.

### 10. Final Summary
Provide a coherent synthesis of the document's most important findings and their relevance to plant protection.

## Important Requirements

- Prioritize scientific accuracy over brevity.
- Do not hallucinate information.
- Do not confuse invasive species with native species.
- Do not confuse confirmed distribution with potential distribution.
- Preserve the relationship between invasive species, host plants, geographical locations, and control methods.
- Do not generate recommendations that are not supported by the document.
- Write the summary in English.
- Use clear headings and structured paragraphs.

INPUT DOCUMENT:

{document_content}"""

In [ ]:

textSummaries,tableSummaries=summaryFromInput(texts=texts,tables=tables,promptText=promptText)

In [ ]:
import uuid
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [ ]:
from langchain_huggingface  import HuggingFaceEmbeddings
def create_multi_vector_retriever(
   vectorstore, text_summaries, texts, table_summaries, tables, image_summaries, images
):
   """
   Create retriever that indexes summaries, but returns raw images or texts
   """

   # Initialize the storage layer
   store = InMemoryStore()
   id_key = "doc_id"

   # Create the multi-vector retriever
   from langchain_ollama import ChatOllama
   model = ChatOllama(
    model="qwen3:4b",
    temperature=0,
    )

   # Helper function to add documents to the vectorstore and docstore
   def add_documents(retriever, doc_summaries, doc_contents):
       doc_ids = [str(uuid.uuid4()) for _ in doc_contents]
       summary_docs = [
           Document(page_content=s, metadata={id_key: doc_ids[i]})
           for i, s in enumerate(doc_summaries)
       ]
       retriever.vectorstore.add_documents(summary_docs)
       retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

   # Add texts, tables, and images
   # Check that text_summaries is not empty before adding
   if text_summaries:
       add_documents(retriever, text_summaries, texts)
   # Check that table_summaries is not empty before adding
   if table_summaries:
       add_documents(retriever, table_summaries, tables)
   # Check that image_summaries is not empty before adding
   if image_summaries:
       add_documents(retriever, image_summaries, images)

   return retriever


# The vectorstore to use to index the summaries
vectorstore = Chroma(
   collection_name="mm_rag_cj_blog",
   embedding_function=HuggingFaceEmbeddings(
       model_name="BAAI/bge-m3",
       encode_kwargs={"normalize_embeddings":True},
   )

)

# Create retriever
retriever= create_multi_vector_retriever(
   vectorstore,
   text_summaries,
   texts,
   table_summaries,
   tables,
   image_summaries,
   img_base64_list,
)

In [ ]:
question=''
output=retriever.invoke(question)